### Model Pipeline

1. Raw table_data (string)

        ↓ linearize

2. Linearized text (string)

        ↓ tokenizer.encode()

3. Token IDs (integers)

        ↓ model's embedding layer (learned lookup table, inside the model)

4. Token embeddings + positional embeddings

        ↓ encoder self-attention layers (bidirectional)

5. Encoder hidden states (contextualized representations, one vector per input token)

        ↓ fed into every decoder layer via cross-attention

6. Decoder (causal self-attention + cross-attention into step 5)

        ↓ generates one token at a time, autoregressively

7. Output logits → softmax → generated token

        ↓ repeat step 6-7 until <eos>
        
8. Decoded text = generated financial commentary

### Installs and Imports

In [ ]:
!pip install -q -U transformers
!pip install -q -U datasets
!pip install -q -U evaluate
!pip install -q "tokenizers>=0.22.0,<0.23.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
!pip install  -q -U rouge_score
!pip install  -q -U nltk
!pip install  -q -U sacrebleu
!pip install  -q -U bert-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
import evaluate
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
import nltk

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
def init_seed(init_random=20):
  random.seed(init_random)
  np.random.seed(init_random)
  torch.manual_seed(init_random)
def sys_info():
  print("PyTorch version:", torch.__version__)
  if torch.cuda.is_available():
    print("GPU Present :", torch.cuda.get_device_name(0))
  else:
    print("CUDA not available !")
def build_encoder_input(row):
    return (
        "Generate a financial market report.\n\n"
        f"Instruction:\n"
        f"{row['instruction']}\n\n"
        f"Market summary:\n"
        f"{row['market_summary']}"
    )
def prep_data(df):
  df["encoder_input"] = df.apply(build_encoder_input,axis=1)
  df["market_summary"] = df["table_data"].apply(summarize_market_table)
  df["decoder_target"] = df["report"]
  df["input_length"] = df["encoder_input"].apply(token_length)
  df["target_length"] = df["decoder_target"].apply(token_length)
  return
def print_ip_op(df, index=0):
  print("Input:",df.iloc[index]["encoder_input"])
  print("Output:",df.iloc[index]["decoder_target"])
  return
def count_percent_by_threshold(df, col, threshold):
  count = (df[col] > threshold).sum()
  percent = (df[col] > threshold).mean()
  print(f'{col} > {threshold}: {count} ({percent:.2%})')
  return

### Setting Execution environment

In [ ]:
# path setup
path1 = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/train_splitEDA.py"
path2 = "/content/drive/MyDrive/GitHub/266-final-project/train_splitEDA.py"
prjfolder = "/content/drive/MyDrive/GitHub/266-final-project/"
if os.path.exists(path1):
  prjfolder = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/"
elif os.path.exists(path2):
  prjfolder = "/content/drive/MyDrive/GitHub/266-final-project/"
prjfolder
save_dir = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/"

In [ ]:
initseed = 20
init_seed(initseed)
sys_info()
init_script = os.path.join(prjfolder, "train_splitEDA.py")
init_script

PyTorch version: 2.11.0+cu128
GPU Present : Tesla T4


'/content/drive/MyDrive/GitHub/266-final-project/train_splitEDA.py'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
%run -i "$init_script"

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (7823 > 512). Running this sequence through the model will result in indexing errors


### Data preparation / formatting

In [ ]:
prep_data(train_df)
prep_data(test_df)

In [ ]:
train_dataset = Dataset.from_pandas(train_df[["encoder_input", "decoder_target"]],
                                    preserve_index=False)

test_dataset = Dataset.from_pandas(test_df[["encoder_input", "decoder_target"]],
                                   preserve_index=False)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['encoder_input', 'decoder_target'],
    num_rows: 3143
})
Dataset({
    features: ['encoder_input', 'decoder_target'],
    num_rows: 795
})


In [ ]:
print_ip_op(train_df, 10)

Input: Generate a financial market report.
Instruction: Please act as an expert financial market analyst. Please generate a market report:
1. by analyzing the historical market data provided.
2. following the market report example provided.

Financial data:
Product Name: Live Cattle Future (front month) (February)
Symbol: LEG2

Date | Open | High | Low | Close | Volume
2021-12-29 | 140.0 | 141.43 | 139.98 | 140.72 | 26245.0
2021-12-30 | 140.28 | 140.62 | 139.85 | 139.98 | 18683.0
2021-12-31 | 139.98 | 139.98 | 139.43 | 139.7 | 15003.0
2022-01-03 | 139.78 | 140.4 | 138.88 | 138.93 | 19752.0
2022-01-04 | 138.65 | 138.72 | 136.78 | 137.82 | 35656.0
2022-01-05 | 138.0 | 138.32 | 136.88 | 137.25 | 24866.0
2022-01-06 | 137.4 | 137.55 | 136.38 | 137.35 | 24741.0
2022-01-07 | 137.35 | 137.7 | 136.57 | 137.32 | 29519.0
2022-01-10 | 136.9 | 137.38 | 136.02 | 136.25 | 35305.0
2022-01-11 | 136.28 | 138.18 | 136.25 | 137.68 | 36728.0
2022-01-12 | 137.75 | 137.85 | 136.4 | 136.57 | 29913.0
2022-01-1

In [ ]:
print_ip_op(test_df)

Input: Generate a financial market report.
Instruction: Please act as an expert financial market analyst. Please generate a market report:
1. by analyzing the historical market data provided.
2. following the market report example provided.

Financial data:
Product Name: Live Cattle Future (front month) (October)
Symbol: LEV2

Date | Open | High | Low | Close | Volume
2022-09-26 | 144.48 | 145.35 | 142.9 | 143.48 | 13746.0
2022-09-27 | 144.22 | 144.32 | 143.15 | 143.57 | 11745.0
2022-09-28 | 144.15 | 144.2 | 142.93 | 143.05 | 11691.0
2022-09-29 | 143.35 | 144.35 | 142.72 | 144.12 | 12670.0
2022-09-30 | 144.12 | 144.5 | 143.22 | 143.28 | 9096.0
2022-10-03 | 143.75 | 144.9 | 143.72 | 144.32 | 8325.0
2022-10-04 | 144.48 | 144.75 | 144.15 | 144.2 | 6274.0
2022-10-05 | 144.35 | 145.18 | 144.02 | 144.68 | 6778.0
2022-10-06 | 145.1 | 145.38 | 144.57 | 145.32 | 7482.0
2022-10-07 | 145.38 | 145.55 | 145.12 | 145.32 | 6105.0
2022-10-10 | 145.45 | 145.72 | 144.15 | 144.7 | 4993.0
2022-10-11 | 145

In [ ]:
train_df[["input_length", "target_length"]].describe()

,input_length,target_length
count,3143.000000,3143.000000
mean,4772.636335,131.379574
std,4057.868174,86.968900
min,1297.000000,6.000000
25%,1824.500000,67.000000
50%,3339.000000,113.000000
75%,4992.000000,169.500000
max,14113.000000,560.000000


In [ ]:
test_df[["input_length", "target_length"]].describe()

,input_length,target_length
count,795.000000,795.000000
mean,5094.729560,148.171069
std,4326.419529,74.627747
min,1535.000000,11.000000
25%,1895.000000,96.000000
50%,3474.000000,139.000000
75%,5277.000000,184.000000
max,14041.000000,424.000000


In [ ]:
count_percent_by_threshold(train_df, "target_length", 384)

target_length > 384: 42 (1.34%)


In [ ]:
count_percent_by_threshold(train_df, "input_length", 256)

input_length > 256: 3143 (100.00%)


##BART Model Training and Evaluation

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# 1. Initialize BART Model and Tokenizer
bart_model_name = "facebook/bart-base"
tokenizer_bart = BartTokenizer.from_pretrained(bart_model_name)
model_bart = BartForConditionalGeneration.from_pretrained(bart_model_name)



vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [ ]:
# 2. Tokenization Function for BART
Max_input_length = 256
Max_target_length = 384

def tokenize_batch_bart(examples):
    model_inputs = tokenizer_bart(
        examples["encoder_input"],
        max_length=Max_input_length,
        truncation=True
    )

    labels = tokenizer_bart(
        text_target=examples["decoder_target"],
        max_length=Max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs





In [ ]:
# 3. Process Datasets
tokenized_train_bart = train_dataset.map(tokenize_batch_bart, batched=True)
tokenized_test_bart = test_dataset.map(tokenize_batch_bart, batched=True)


Map:   0%|          | 0/3143 [00:00<?, ? examples/s]

Map:   0%|          | 0/795 [00:00<?, ? examples/s]

Setup till here for loaded model

In [ ]:
# 4. Data Collator and Training Arguments
data_collator_bart = DataCollatorForSeq2Seq(tokenizer=tokenizer_bart, model=model_bart)

training_args_bart = Seq2SeqTrainingArguments(
    output_dir="./bart-financial-commentary",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    report_to="none"
)


In [ ]:
# 5. Initialize Trainer
trainer_bart = Seq2SeqTrainer(
    model=model_bart,
    args=training_args_bart,
    train_dataset=tokenized_train_bart,
    eval_dataset=tokenized_test_bart,
    processing_class=tokenizer_bart,
    data_collator=data_collator_bart
)

print("BART Trainer initialized. Run trainer_bart.train() to begin training.")

BART Trainer initialized. Run trainer_bart.train() to begin training.


In [ ]:
# 6 Training
trainer_bart.train()

Epoch,Training Loss,Validation Loss
1,No log,2.441552
2,2.170095,2.445727
3,2.022060,2.429555
4,1.974116,2.411778
5,1.974116,2.411976


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1965, training_loss=2.035134507196247, metrics={'train_runtime': 699.0773, 'train_samples_per_second': 22.48, 'train_steps_per_second': 2.811, 'total_flos': 2395502110310400.0, 'train_loss': 2.035134507196247, 'epoch': 5.0})

In [ ]:
# evaluate on test set
test_metrics = trainer_bart.evaluate(
    max_length=Max_target_length,
    num_beams=4
)

print(test_metrics)

Training Loss,Validation Loss,Epoch
No log,2.408306,0


{'eval_loss': 2.4083058834075928}


### Save BART Model and Tokenizer
This cell saves the fine-tuned BART model and tokenizer to a persistent directory on Google Drive.

In [ ]:
import os

# Define the save path for BART
bart_model_dir = os.path.join(save_dir, "bart_base_model1")

# Save model and tokenizer
trainer_bart.save_model(bart_model_dir)
tokenizer_bart.save_pretrained(bart_model_dir)

print(f"BART model and tokenizer saved to: {bart_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BART model and tokenizer saved to: /content/drive/MyDrive/Colab Notebooks/266/266_final_project/bart_base_model1


### Load BART Model (Standalone Session)
Run this cell to load your saved BART model. It includes all necessary imports to start directly from here.

In [ ]:
import os
import torch
from transformers import BartTokenizer, BartForConditionalGeneration

# 1. Define the directory path (ensure your Drive is mounted if using Colab)
# Adjust the path below if your save_dir or folder structure differs
save_dir = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/"
bart_model_dir = os.path.join(save_dir, "bart_base_model1")

# 2. Load the objects
print(f"Loading BART model from {bart_model_dir}...")
loaded_tokenizer_bart = BartTokenizer.from_pretrained(bart_model_dir)
loaded_model_bart = BartForConditionalGeneration.from_pretrained(bart_model_dir)

# 3. Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
loaded_model_bart.to(device)

print("BART model and tokenizer loaded successfully.")

Loading BART model from /content/drive/MyDrive/Colab Notebooks/266/266_final_project/bart_base_model1...


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

BART model and tokenizer loaded successfully.


In [ ]:
!pip install -q rouge_score bert_score

In [ ]:
import evaluate
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
import nltk

# 1. Ensure METEOR and NLTK dependencies are ready
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)
meteor = evaluate.load('meteor')

from sklearn.model_selection import KFold
import sys
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
import evaluate
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader

# 1. Load Evaluation Metrics
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')
sacrebleu = evaluate.load('sacrebleu')

# 2. Generation Loop using Loaded Model
def get_model_predictions(model, tokenizer, dataset, batch_size=16):
    model.eval()
    predictions = []
    references = []

    dataloader = DataLoader(dataset, batch_size=batch_size)

    print(f"Generating predictions for {len(dataset)} samples...")
    with torch.no_grad():
        for batch in dataloader:
            inputs = tokenizer(
                batch['encoder_input'],
                padding=True,
                truncation=True,
                max_length=Max_input_length,
                return_tensors="pt"
            ).to(device)

            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=Max_target_length,
                num_beams=4,
                early_stopping=True
            )

            preds = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
            labels = batch['decoder_target']

            predictions.extend(preds)
            references.extend(labels)

    return predictions, references

# Execute generation
decoded_preds_bart, decoded_labels_bart = get_model_predictions(
    loaded_model_bart,
    loaded_tokenizer_bart,
    test_dataset
)

# 3. Compute Metrics
rouge_results_bart = rouge.compute(predictions=decoded_preds_bart, references=decoded_labels_bart)
sacrebleu_results_bart = sacrebleu.compute(predictions=decoded_preds_bart, references=[[r] for r in decoded_labels_bart])

print("\n--- BART Lexical Metrics ---")
for k, v in rouge_results_bart.items():
    print(f"{k}: {v:.4f}")
print(f"sacrebleu: {sacrebleu_results_bart['score']:.4f}")

# 4. Compute Semantic Metrics (BERTScore)
bertscore_results_bart = bertscore.compute(predictions=decoded_preds_bart, references=decoded_labels_bart, lang='en')
print("\n--- BART Semantic Metrics (BERTScore) ---")
print(f"Precision: {np.mean(bertscore_results_bart['precision']):.4f}")
print(f"Recall: {np.mean(bertscore_results_bart['recall']):.4f}")
print(f"F1: {np.mean(bertscore_results_bart['f1']):.4f}")

Generating predictions for 795 samples...

--- BART Lexical Metrics ---
rouge1: 0.2937
rouge2: 0.1047
rougeL: 0.2037
rougeLsum: 0.2034
sacrebleu: 6.7030


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- BART Semantic Metrics (BERTScore) ---
Precision: 0.8765
Recall: 0.8537
F1: 0.8647


### Adding METEOR Evaluation
To compute the METEOR score, we need the `nltk` library and the `meteor` metric from the `evaluate` package.

In [ ]:


# 2. Safety check: Regenerate BART predictions if variables were lost during env reset
if 'decoded_preds_bart' not in locals():
    print("Regenerating BART predictions for evaluation...")
    # This assumes trainer_bart and tokenized_test_bart exist from earlier successful cells
    predictions_output_bart = trainer_bart.predict(tokenized_test_bart)

    label_ids = predictions_output_bart.label_ids
    label_ids = np.where(label_ids != -100, label_ids, tokenizer_bart.pad_token_id)

    preds = predictions_output_bart.predictions
    preds = np.where(preds != -100, preds, tokenizer_bart.pad_token_id)

    decoded_preds_bart = tokenizer_bart.batch_decode(preds, skip_special_tokens=True)
    decoded_labels_bart = tokenizer_bart.batch_decode(label_ids, skip_special_tokens=True)

# 3. Compute METEOR
meteor_results = meteor.compute(predictions=decoded_preds_bart, references=decoded_labels_bart)
print(f"\n--- Final BART Evaluation ---")
print(f"METEOR Score: {meteor_results['meteor']:.4f}")


--- Final BART Evaluation ---
METEOR Score: 0.2044


### Post-hoc K-Fold Cross Validation on Test Set
This section splits the existing test set into K folds, evaluates each fold independently, and reports the mean and standard deviation of the metrics.

In [ ]:
from sklearn.model_selection import KFold
import numpy as np
import sys
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments

def get_eval_trainer(model, tokenizer):
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
    return Seq2SeqTrainer(
        model=model,
        args=Seq2SeqTrainingArguments(
            output_dir="./temp_eval",
            predict_with_generate=True,
            fp16=torch.cuda.is_available(),
            per_device_eval_batch_size=8,
            report_to="none"
        ),
        data_collator=data_collator,
        processing_class=tokenizer
    )

def run_kfold_evaluation(dataset, model, tokenizer, k=5):
    eval_trainer = get_eval_trainer(model, tokenizer)
    kf = KFold(n_splits=k, shuffle=True, random_state=initseed)
    all_results = []
    indices = np.arange(len(dataset))

    print(f"Starting optimized {k}-fold cross-validation with SacreBLEU...\n")

    for fold, (_, fold_indices) in enumerate(kf.split(indices)):
        fold_dataset = dataset.select(fold_indices)
        print(f"Fold {fold+1}/{k} processing:")

        # Use Trainer's predict
        output = eval_trainer.predict(fold_dataset)

        # Same-line progress feedback
        sys.stdout.write(f"\rFold {fold+1} Status: [DONE] 100% complete")
        sys.stdout.flush()
        print("\nComputing metrics...")

        label_ids = np.where(output.label_ids != -100, output.label_ids, tokenizer.pad_token_id)
        preds = np.where(output.predictions != -100, output.predictions, tokenizer.pad_token_id)

        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        r = rouge.compute(predictions=decoded_preds, references=decoded_labels)
        sb = sacrebleu.compute(predictions=decoded_preds, references=[[r] for r in decoded_labels])
        bs = bertscore.compute(predictions=decoded_preds, references=decoded_labels, lang='en')
        m = meteor.compute(predictions=decoded_preds, references=decoded_labels)

        fold_metrics = {
            'rougeL': r['rougeL'],
            'sacrebleu': sb['score'],
            'bert_f1': np.mean(bs['f1']),
            'meteor': m['meteor']
        }

        all_results.append(fold_metrics)
        print(f"Fold {fold+1} Results -> SacreBLEU: {fold_metrics['sacrebleu']:.4f} | METEOR: {fold_metrics['meteor']:.4f}\n")

    metrics_summary = {}
    for key in all_results[0].keys():
        values = [res[key] for res in all_results]
        metrics_summary[key] = {'mean': np.mean(values), 'std': np.std(values)}

    return metrics_summary

# Run the optimized evaluation
bart_kfold_summary = run_kfold_evaluation(tokenized_test_bart, loaded_model_bart, loaded_tokenizer_bart, k=5)

Starting optimized 5-fold cross-validation with SacreBLEU...

Fold 1/5 processing:


Fold 1 Status: [DONE] 100% complete
Computing metrics...
Fold 1 Results -> SacreBLEU: 0.0134 | METEOR: 0.0694

Fold 2/5 processing:


Fold 2 Status: [DONE] 100% complete
Computing metrics...
Fold 2 Results -> SacreBLEU: 0.0101 | METEOR: 0.0702

Fold 3/5 processing:


Fold 3 Status: [DONE] 100% complete
Computing metrics...
Fold 3 Results -> SacreBLEU: 0.0104 | METEOR: 0.0642

Fold 4/5 processing:


Fold 4 Status: [DONE] 100% complete
Computing metrics...
Fold 4 Results -> SacreBLEU: 0.0121 | METEOR: 0.0641

Fold 5/5 processing:


Fold 5 Status: [DONE] 100% complete
Computing metrics...
Fold 5 Results -> SacreBLEU: 0.0158 | METEOR: 0.0669



In [ ]:
print("\n--- BART K-Fold Cross-Validation Summary ---")
for metric, stats in bart_kfold_summary.items():
    name = "SACREBLEU" if metric == "sacrebleu" else metric.upper()
    print(f"{name}: {stats['mean']:.5f} (+/- {stats['std']:.5f})")


--- BART K-Fold Cross-Validation Summary ---
ROUGEL: 0.10714 (+/- 0.00274)
SACREBLEU: 0.01237 (+/- 0.00208)
BERT_F1: 0.84604 (+/- 0.00111)
METEOR: 0.06696 (+/- 0.00255)


### STS Evaluation with DistilGPT2
This section uses a decoder-only model (DistilGPT2) to compute Semantic Textual Similarity (STS) between the generated reports and the reference reports.

In [ ]:
from transformers import AutoModel, AutoTokenizer
from torch.nn.functional import cosine_similarity

# 1. Load DistilGPT2 for STS
sts_model_name = "distilgpt2"
sts_tokenizer = AutoTokenizer.from_pretrained(sts_model_name)
sts_model = AutoModel.from_pretrained(sts_model_name).to(device)
sts_tokenizer.pad_token = sts_tokenizer.eos_token

def get_embeddings(text_list, model, tokenizer):
    model.eval()
    inputs = tokenizer(text_list, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        # Use mean pooling of hidden states as sentence representation
        embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

def compute_sts_score(predictions, references, batch_size=16):
    scores = []
    print(f"Computing STS scores for {len(predictions)} samples...")

    for i in range(0, len(predictions), batch_size):
        batch_preds = predictions[i:i+batch_size]
        batch_refs = references[i:i+batch_size]

        emb_preds = get_embeddings(batch_preds, sts_model, sts_tokenizer)
        emb_refs = get_embeddings(batch_refs, sts_model, sts_tokenizer)

        # Compute cosine similarity
        sim = cosine_similarity(emb_preds, emb_refs)
        scores.extend(sim.cpu().tolist())

    return np.mean(scores)

# 2. Run STS Evaluation
bart_sts_score = compute_sts_score(decoded_preds_bart, decoded_labels_bart)
print(f"\nBART DistilGPT2 STS Score: {bart_sts_score:.4f}")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Computing STS scores for 795 samples...

BART DistilGPT2 STS Score: 0.9912
